# A Simple Machine Learning Workflow 

This notebook will guide you through a basic supervised machine learning workflow.

> 💡 If you have brought your own dataset, try applying these steps to it at the end of this notebook. 

<img src="../images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">
<img src="../images/02_enriching.png" alt="Enriching diagram" style="max-width: 150px;">
<img src="../images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">
<img src="../images/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">
<img src="../images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">


You can **choose** to work with either of the following datasets: 

- The [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) - a collection of 50,000 labeled IMDB reviews for binary sentiment classification.

- The [NELA-PS dataset](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YHWTFC) -  a collection of 'pink slime' partisan news articles (from which we will use a small sample).

Your **goal** is to train a basic supervised machine learning model to classify text for these datasets: 
- For the Large Movie Review Dataset, the objective is to **predict whether a movie review is positive or negative**.
- For the NELA-PS dataset, the objective is to **predict the outlet that published a given article**.

You will do this by training the model on the 'train' subset of the dataset, and then evaluating its performance on the 'test' subset.

> ⚠️ Remember: the pre-processing, enriching and vectorization steps can be applied in many different ways. Experimentation is very important. On Day 4, you will learn how to systematically perform this experimentation with different choices.

> 💡 The eagle-eyed among you will notice that we have not included the Dimensional Reduction and Clustering 'lego brick' of the course pipeline, in this workbook. That is because here we are training a simple model, using an equally simple Bag-of-Words representation of the text data. As such, this is not required (yet). Tomorrow (Day 3), we will explore Dimentional Reduction and Clustering in detail. 


# 0. Setup 

In [ ]:
#packages you have seen already - 
import os #for os operations
from glob import glob #for filepath operations
import pandas as pd #for dataframes
import numpy as np #for numerical operations

#some new packages -
import re #for regex operations (used in pre-processing)
import spacy #for the 'enrich step' - Part-of-Speech tagging and Named Entity Recognition (NER)

#packages required for our ML pipeline
from sklearn.preprocessing import LabelEncoder #for encoding labels as numbers
from sklearn.model_selection import train_test_split #for splitting data into training and test sets
from sklearn.feature_extraction.text import CountVectorizer #a vectorizer (converts text to numbers)
from sklearn.naive_bayes import MultinomialNB #the model (a Naive Bayes classifier)
from sklearn.metrics import confusion_matrix, classification_report #for model evaluation

## 1. Preprocessing  
<img src="../images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 150px;">

In this step, we will load in the data, explore it, and clean it up a bit. 

> 💡 As you have seen this-morning, vectorization (which comes after pre-processing) can actually take care of a lot of pre-processing for you (e.g., stopword removal, lemmatization, etc). However, there are two reasons why pre-processing remans important:
- The first is an **input reason**: your text might not be very 'clean' to begin with. It might contain non-unicode characters or html tags (e.g. <\/br>) that will confuse the vectorizer. 
- The second it an **output reason**: perhaps you want something different or targeted than a verbatim copy of the text, that is motivated by theoretical / reserach question reasons. 

⚠️ Therefore, think carefully about what pre-processing steps you want to apply, and why. You can always come back and change this part of the pipeline at any time. Do experiment. 


In [ ]:
#first, let's define where this dataset is stored on your computer:
datadir = '/Users/rupertkiddle/Downloads/'

#load the data into a df using .csv: 
df_nela = pd.read_csv(os.path.join(datadir, 'nela_data.csv'))

df_nela.head(3) #preview the first three rows of the dataframe

In [ ]:
#MISSING DATA - 
#i.e., check if any rows have missing data, and remove them if so.
print(df_nela.isnull().sum()) #check how many missing values there are in each column

#remove rows with missing values
df_nela = df_nela.dropna()

In [ ]:
#print first row of text: 
print(df_nela['text'][0])

In [ ]:
#TEXT NORMALIZATION - 
#i.e., ensuring the text is in a standard format.

#Let's inspect the text before and after each cleaning step:
print(f"first row of text before cleaning:\n{df_nela['text'][0]}")

#first, let's begin  by normailizing the text - lowercasing, stripping lead/tail whitespace.
df_nela['text'] = df_nela['text'].str.lower().str.strip()
print(f"first row of text after lowercasing, stripping whitespace:\n{df_nela['text'][0]}")

#then, let's remove any internal whitespace (e.g., new lines, tabs, multiple spaces)
df_nela['text'] = df_nela['text'].apply(lambda s: ' '.join(str(s).split()))

#We can also remove any non-UTF-8 characters with the following line:
df_nela['text'] = df_nela['text'].apply(lambda x: x.encode('utf-8', 'ignore').decode('utf-8'))
print(f"first row of text after removing non-UTF-8 characters:\n{df_nela['text'][0]}")

#Let's also check if there are any html tags in the text:
#regex pattern: begins and ends with < and >; anything in between (*), non greedy (?)
def remove_html_tags(text):
    clean = re.compile('<.*?>') 
    return re.sub(clean, '', text)
df_nela['text'] = df_nela['text'].apply(remove_html_tags)
print(f"first row of text after removing html tags:\n{df_nela['text'][0]}")


In [ ]:
#PREVENTING LABEL LEAKAGE -
#i.e., ensuring that the text does not contain the label we are trying to predict. 

#first, let's get the unique labels in the dataset:
print(df_nela['outlet'].unique())
#we can see that the labels are the names of the news outlets.

#print how many times each outlet appears in the text column:
for outlet in df_nela['outlet'].unique():
    count = df_nela['text'].str.contains(outlet.lower()).sum()
    print(f"{outlet}: {count} instances in text")

#Let's remove those outlet names from the text:
for outlet in df_nela['outlet'].unique():
    df_nela['text'] = df_nela['text'].str.replace(outlet.lower(), '')

# 2. Enriching
<img src="../images/02_enriching.png" alt="Enriching diagram" style="max-width: 150px;">

In this step, we will 'tag' our tokens with additional information, using Part-of-Speech (POS) tagging and Named Entity Recognition (NER).

> 💡 This step }is not strictly necessary. If you are short on time, skip it, and come back later. 

In [ ]:
#PERFORMING PoS TAGGING and NER - 

#load spacy model (download if not present)
nlp = spacy.load('en_core_web_sm')

# function returns a tuple: (pos_tags_list, entities_list)
def enrich_text(text):
    doc = nlp(text)
    pos_tags = [token.pos_ for token in doc]
    # store entities as tuples of (text, label) for clarity
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return pos_tags, entities


#since this is just for illustration, let's just process first 5 rows. 
df_nela_sample5 = df_nela.head(5).copy()

#apply to the 'text' column, and create two new columns: 'pos_tags' and 'entities'
df_nela_sample5[['pos_tags', 'entities']] = df_nela_sample5['text'].apply(lambda x: pd.Series(enrich_text(x)))


In [ ]:
#EXPLORING THE PoS TAGS and NER - 
#NOTE: the above code added two new columns (pos_tags and entities)

#let's take a look at the output: 
print(df_nela_sample5[['text', 'pos_tags', 'entities']]) 

> 💡 `POS-tagging` is very useful if you want to identify certain types of phrases within your texts (e.g., NOUN+ADJ like "angry protestors").

> 💡 `Named Entity Recognition` is useful if you want to identify mentions of specific entities (e.g., people, organisations, locations, etc).

# 3. Vectorization
<img src="../images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 150px;">

In this step, we will convert our text data into a numerical format that can be used by machine learning algorithms. 

In [ ]:
#OPTION 1: COUNT VECTORIZER -

#Let's define a CountVectorizer with some parameters (you can experiment with these):
vectorizer_CV = CountVectorizer(lowercase=False, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.8)

#fit the vectorizer to the enriched text, and transform the text to a document-term matrix
X_CV = vectorizer_CV.fit_transform(df_nela['text'])

#print the number of terms (i.e., columns) in the document-term matrix
print(f"CountVectorizer - number of features: {len(vectorizer_CV.get_feature_names_out())}")

#print the number of documents (i.e., rows) in the document-term matrix:
print(f"CountVectorizer - number of documents: {X_CV.shape[0]}")

In [ ]:
#OPTION 2: TF-IDF VECTORIZER -

#Let's define a TfidfVectorizer with some parameters (you can experiment with these):
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer_TFIDF = TfidfVectorizer(lowercase=False, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.8)

#fit the vectorizer to the enriched text, and transform the text to a document-term matrix
X_TFIDF = vectorizer_TFIDF.fit_transform(df_nela['text'])

#print the number of features (i.e., unique tokens) in the document-term matrix
print(f"TfidfVectorizer - number of features: {len(vectorizer_TFIDF.get_feature_names_out())}")

#print the number of documents (i.e., rows) in the document-term matrix:
print(f"TfidfVectorizer - number of documents: {X_TFIDF.shape[0]}")

In [ ]:
#WHAT IS THE DIFFERENCE BETWEEN COUNT VECTORIZER AND TF-IDF VECTORIZER?
#Q: what is the highest and lowest weighted word in each document-term matrix, and it's value?
#NOTE: This is for illustrative purposes, you do not need to understand this code in detail. 

def analyze_vectorizer(X, vectorizer, name):
    """Analyze top 5 and bottom 5 words for the first document only"""
    # Convert to array and get values for the first document (row 0)
    X_array = X.toarray()
    first_doc_values = X_array[0, :]
    
    # Get feature names
    features = vectorizer.get_feature_names_out()
    
    # Get indices for top 5 and bottom 5 (excluding zeros)
    non_zero_mask = first_doc_values > 0
    non_zero_values = first_doc_values[non_zero_mask]
    non_zero_features = features[non_zero_mask]
    
    # Sort by values
    sorted_indices = np.argsort(non_zero_values)
    
    print(f"\n{name} - First Document:")
    print("Top 5 words:")
    for i in range(-1, -6, -1):  # Last 5 (highest)
        idx = sorted_indices[i]
        print(f"  '{non_zero_features[idx]}': {non_zero_values[idx]:.4f}")
    
    print("Bottom 5 words:")
    for i in range(5):  # First 5 (lowest non-zero)
        idx = sorted_indices[i]
        print(f"  '{non_zero_features[idx]}': {non_zero_values[idx]:.4f}")

# Analyze both vectorizers
analyze_vectorizer(X_CV, vectorizer_CV, "CountVectorizer")
analyze_vectorizer(X_TFIDF, vectorizer_TFIDF, "TF-IDF Vectorizer")

#and print the text of the article for comparison: 
print(f"\nText of the first document:\n{df_nela['text'].iloc[0]}")

# 4. Modelling
<img src="../images/05_modelling.png" alt="Modelling diagram" style="max-width: 150px;">

In this step, we will train a simple supervised machine learning model (Multinomial Naive Bayes) on the 'training' data subset. 

Remember, we refer to to: 
- The 'features' as **X** (the input data, i.e., the vectorized text data)
- The 'labels' as **y** (the output data, i.e., the sentiment labels for the movie reviews, or the outlet labels for the news articles)

In [ ]:
#We need to encode the labels (i.e., the news outlets) as numbers: 
le = LabelEncoder()
y = le.fit_transform(df_nela['outlet']) #encode the labels as numbers

#print what outlets they correspond to:
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
#next , we will split the data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X_CV, y, test_size=0.2, random_state=42, stratify=y)

#Now, we everything is ready to train a model!

#We will use a simple Multinomial Naive Bayes classifier:
model = MultinomialNB() #here we instantiate (create) the model so that we can use it. 

#fit the model to the training data:
model.fit(X_train, y_train) #giving it the training data (X_train) and the labels (y_train).DS_Store

#predict the labels for the test data:
#NOTE: we need the true labels (y_test) out to evaluate the model in the next step.
y_pred = model.predict(X_test) #predict the labels for the test data (X_test)

# 5. Evaluation
<img src="../images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 150px;">

In this step, we will evaluate the performance of our trained model, b

In [ ]:
#we can also visualize the confusion matrix
confusion_matrix_df = pd.DataFrame(confusion_matrix(y_test, y_pred), index=le.classes_, columns=le.classes_)
confusion_matrix_df

In [ ]:
#the classification report gives us precision, recall, f1-score for each class (i.e., each label)
print(classification_report(y_test, y_pred))

#print our labels again for reference:
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

#for now, higher = better (we will cover these metrics on D4).DS_Store
#if curious: 
#Precision = TP / (TP + FP) — of predicted positives, how many are correct?
#Recall = TP / (TP + FN) — of actual positives, how many did you find?
#F1 = harmonic mean of precision and recall.
#support = number of occurances of the class (the label).

> 💡 How did it perform? Better, or worse, for certain classes (labels)? Consider returning to your pre-processing and vectorization steps, to see the effects of different choices.

# 6. "All Together Now" (optional)
<img src="../images/01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 120px;">
<img src="../images/03_vectorization.png" alt="Vectorization diagram" style="max-width: 120px;">
<img src="../images/05_modelling.png" alt="Modelling diagram" style="max-width: 120px;">
<img src="../images/06_evaluation.png" alt="Evaluation diagram" style="max-width: 120px;">

Sklearn's Pipeline() allows us to combine multiple steps into a single object. This is very useful for automating the process of building and evaluating models, especially when we want to systematically test different configurations (e.g., different pre-processing steps, vectorizers, models, etc). 

In [ ]:
#import it like this: 
from sklearn.pipeline import Pipeline

#we can then create a pipeline with our vectorizer and model:
#NOTE: mouse over the Pipeline() function to see what else you can add.
my_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(lowercase=False, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.8)),
    ('model', MultinomialNB())
])

#split the data into training and test sets again:
#NOTE: this time we are passing the raw text data (df_nela['text']).
X_text = df_nela['text']
X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)

#fit the pipeline to the training data:
my_pipeline.fit(X_train, y_train)

#predict the labels for the test data:
y_pred = my_pipeline.predict(X_test)

> 💡 That's it! Pipeline just allows us to streamline the process of building and evaluating our model by encapsulating all the steps into a single object. This becomes essential for automatically testing different configurations to see what performs best - a 'grid search' - which we will explore on Day 4. 

# 7. Try on your own data (optional)

In [ ]:
#...